In [ ]:
# Importing the required libraries

import torch
import torch.nn as nn
import pandas as pd
import nltk
from nltk.corpus import stopwords
from collections import Counter
from nltk.tokenize import word_tokenize
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creating the dataframe using pandas
df = pd.read_csv('reviews.csv')
# print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
# Creating train and test dataset

x,y = df['review'].values,df['sentiment'].values
x_train,x_test,y_train,y_test = train_test_split(x,y,stratify=y)

In [ ]:
# Checking the percentage of the uppercase words to determine wheter lowercasing should be applied or not


# Combine all text into one string
all_text = ' '.join(df['review'].astype(str))

# Find all words
words = re.findall(r'\b[A-Z]{2,}\b', all_text)  # Words with 2 or more all-uppercase letters

# Count them
capital_words_count = len(words)
word_count = len(all_text)
print(f"total words: {word_count}")
print(f"Total ALL CAPS words: {capital_words_count}")

# Calculate percentage
Percent_of_uppercase = (capital_words_count/word_count)*100
print(f"Percentage of uppercase words :{Percent_of_uppercase}")

total words: 65521550
Total ALL CAPS words: 82831
Percentage of uppercase words :0.1264179495143201


Percentage of uppecase words is very low so we can apply lowercasing.

In [ ]:
# processing the string
# removing the puntuation, whitespace, character etc

def preprocess_string(s):
    # Remove all non-word characters
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)

    return s

# Downloading the stopwords from nltk
nltk.download('stopwords')

# Tockenizeing and padding the reviews
def tockenize(x_train,y_train,x_val,y_val):
  word_list = []

  # Removing the stop words
  stop_words = set(stopwords.words('english'))
  for sentence in x_train:
    for word in sentence.lower().split():
      word = preprocess_string(word)
      if word not in stop_words and word != '':
        word_list.append(word)

  # Getting the most common 1100 words

  corp = Counter(word_list)            # Creates a dictionary of words and their count
  common = corp.most_common(1100)
  #print(common)
  corpus = []
  for i, j in common:
    corpus.append(i)

  # Creating a dictionary for vocablary and assigning them an integer
  onehot_dict = {w:i for i, w in enumerate(corpus, start=1)}

  # Tokenizing

  final_x_train, final_x_test = [],[]
  # Taking the words from corpus which are in our vocablury and making training dataset
  for sentence in x_train:
    temp_list = []
    for word in sentence.lower().split():
      if preprocess_string(word) in onehot_dict.keys():
        temp_list.append(onehot_dict[preprocess_string(word)])
    final_x_train.append(temp_list)

  # Taking the words from corpus which are in our vocablury and making testing dataset
  for sentence in x_val:
    temp_list = []
    for word in sentence.lower().split():
      if preprocess_string(word) in onehot_dict.keys():
        temp_list.append(onehot_dict[preprocess_string(word)])
    final_x_test.append(temp_list)

  # Padding the train data
  train_pad = np.zeros((len(final_x_train), 1600),dtype=int)
  for i, review in enumerate(final_x_train):
    if len(review) != 0:
      train_pad[i, -len(review):] = np.array(review)[:1600]

  # Padding the test data
  test_pad = np.zeros((len(final_x_test), 1600),dtype=int)
  for i, review in enumerate(final_x_test):
    if len(review) != 0:
      test_pad[i, -len(review):] = np.array(review)[:1600]


  # Assigning 1 for positive and 0 for negative
  encoded_train = [1 if label=='positive' else 0 for label in y_train]
  encoded_test = [1 if label=='positive' else 0 for label in y_val]

  return np.array(train_pad), np.array(test_pad), np.array(encoded_train), np.array(encoded_test), onehot_dict

x_train, x_test, y_train, y_test, vocab = tockenize(x_train,y_train,x_test,y_test)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Dataset and dataloading

train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test))

batch_size = 50

# Creating Dataloader
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(test_dataset, shuffle=True, batch_size=batch_size)

In [ ]:
# Creating the neural network

class SentimentAnalysisModel(nn.Module):
    def __init__(self, output_dim, num_layer, hidden_dim, embd_dim, vocab_size, drop_prb = 0.5):
      super(SentimentAnalysisModel, self).__init__()
      self.output_dim = output_dim
      self.num_layer = num_layer
      self.hidden_dim = hidden_dim

      self.vocab_size = vocab_size

      # Creating a the embedding layer
      # Converting the one hot encoded vector in dense vectors
      self.embedding = nn.Embedding(vocab_size, embd_dim)

      # creating the lstm layer
      self.lstm = nn.LSTM(input_size=embd_dim, hidden_size = self.hidden_dim, num_layers=num_layer, batch_first = True)

      self.dropout = nn.Dropout(0.3)

      self.lin_lay = nn.Linear(hidden_dim, output_dim)


    def forward(self,x,hidden):
        batch_size = x.size(0)
        # embeddings and lstm_out
        embeds = self.embedding(x)
        #print(embeds.shape)
        lstm_out, hidden = self.lstm(embeds, hidden)

        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim)

        # dropout and fully connected layer
        out = self.dropout(lstm_out)
        out = self.lin_lay(out)

        out = out.view(batch_size, -1)


        out = out[:, -1]  # get last time step
        return out, hidden  # return raw logits


    def init_hidden(self, batch_size):
        # Initializes hidden state
        # Create two new tensors with sizes n_layers x batch_size x hidden_dim,
        # initialized to zero, for hidden state and cell state of LSTM
        h0 = torch.zeros((self.num_layer,batch_size,self.hidden_dim)).to(device)
        c0 = torch.zeros((self.num_layer,batch_size,self.hidden_dim)).to(device)
        hidden = (h0,c0)
        return hidden

In [ ]:
num_layers = 2
vocab_size = len(vocab) + 1 #extra 1 for padding
embedding_dim = 150      # length of the dense vector
output_dim = 1
hidden_dim = 256  # No of nodes in RNN

# Creating the model
model = SentimentAnalysisModel(output_dim,num_layers, hidden_dim, embedding_dim, vocab_size, drop_prb=0.5)

#moving to gpu
model.to(device)

print(model)

SentimentAnalysisModel(
  (embedding): Embedding(1101, 150)
  (lstm): LSTM(150, 256, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (lin_lay): Linear(in_features=256, out_features=1, bias=True)
)


In [ ]:
# Training
lr = 0.001                      #learning rate
criterion = nn.BCEWithLogitsLoss()    # binary cross entrop with logits loss function (no need to use sigmoid)

#using adam optimezer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

#Accuracy function
def acc(logits, label):
    probs = torch.sigmoid(logits.squeeze())
    pred = torch.round(probs)
    return torch.sum(pred == label.squeeze()).item()


clip = 5
epochs = 6
valid_loss_min = np.inf

epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]
# Training loop
for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    model.train()
    # initialize hidden state
    h = model.init_hidden(batch_size)

    # training
    for inputs, labels in train_loader:

        inputs, labels = inputs.to(device), labels.to(device)
        # Creating new variables for the hidden state, otherwise
        # we'd backprop through the entire training history
        h = tuple([each.data for each in h])

        model.zero_grad()


        output,h = model(inputs,h)

        # calculate the loss and perform backpropogation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        # calculating accuracy
        accuracy = acc(output,labels)
        train_acc += accuracy

        # preventing the exploding gradient problem
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()



    val_h = model.init_hidden(batch_size)
    val_losses = []
    val_acc = 0.0
    model.eval()

    # validating
    for inputs, labels in valid_loader:
            val_h = tuple([each.data for each in val_h])

            inputs, labels = inputs.to(device), labels.to(device)

            output, val_h = model(inputs, val_h)
            val_loss = criterion(output.squeeze(), labels.float())

            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy

    # calculating the losses and accuracy
    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/len(train_loader.dataset)
    epoch_val_acc = val_acc/len(valid_loader.dataset)

    # updating the list of accuracy and losses in each epoch
    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)
    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    # Save the model which have the best accuracy
    if epoch_val_loss <= valid_loss_min:
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(
        valid_loss_min,
        epoch_val_loss))
        torch.save(model.state_dict(), 'best_model.pt')
        valid_loss_min = epoch_val_loss

Epoch 1
train_loss : 0.48330514458815255 val_loss : 0.40750046288967134
train_accuracy : 77.16 val_accuracy : 82.6
Validation loss decreased (inf --> 0.407500). Saving model ...
Epoch 2
train_loss : 0.3413844936092695 val_loss : 0.32532503074407576
train_accuracy : 85.48266666666666 val_accuracy : 85.98400000000001
Validation loss decreased (0.407500 --> 0.325325). Saving model ...
Epoch 3
train_loss : 0.2944538092017174 val_loss : 0.3238613916635513
train_accuracy : 87.90133333333333 val_accuracy : 86.512
Validation loss decreased (0.325325 --> 0.323861). Saving model ...
Epoch 4
train_loss : 0.25365421605110167 val_loss : 0.33682209181785583
train_accuracy : 89.712 val_accuracy : 86.68
Epoch 5
train_loss : 0.19919363782306512 val_loss : 0.3654236532151699
train_accuracy : 92.14933333333335 val_accuracy : 85.14399999999999
Epoch 6
train_loss : 0.12418096778541804 val_loss : 0.45967832492291927
train_accuracy : 95.344 val_accuracy : 84.86399999999999
